# qust monitor 用法：从输入列到图形输出

[项目地址](https://baiguoname.github.io/qust/site) · [git地址](https://github.com/baiguoname/qust)


monitor 是 qust 的可视化层。你可以把它理解成：

```text
qust 表达式算出绘图数据 -> monitor 根据图类型解释列 -> runtime.plot(data) 打开交互图
```

示例不用 matplotlib，图形由 qust 原生 monitor 输出。

学习重点：

1. 每种图需要什么输入列；
2. `monitor("slot")` 的 slot 是什么；
3. 多个图层怎么叠到同一个 slot；
4. `make_monitor + add_grid` 怎么组织 dashboard；
5. violin 为什么推荐先算 `batch.violin_profile()`。


In [1]:
import qust as qs

from qust import col, pms
from qust._polars import pl
import qust.datasource as qds

pl.Config.set_tbl_rows(16)
pl.Config.set_tbl_cols(14)

DATA_KLINE = "https://github.com/baiguoname/qust/blob/main/examples/data/data_kline3.parquet?raw=true"
DATA_FUTURE = "https://github.com/baiguoname/qust/blob/main/examples/data/kline_data_all.parquet?raw=true"
DATA_STOCK = "https://github.com/baiguoname/qust/blob/main/examples/data/stock_data_kline.parquet?raw=true"

from qust.monitor import mark_shape


## 1. 准备绘图数据

所有 monitor 图都不是凭空画的，它只消费表达式输出的列。

下面先构造一张统一的绘图表：

- `datetime`：x 轴；
- `open/high/low/close/volume`：K 线；
- `ma30`：30 窗口均线；
- `ret`：单 bar 收益；
- `up`：是否上涨，用来画 mark。


In [2]:
raw_kline = pl.read_parquet(DATA_KLINE).sort(["ticker", "ct", "datetime"])
PLOT_ROW = raw_kline.select("ticker", "ct").unique().sort(["ticker", "ct"]).row(0, named=True)
PLOT_TICKER = PLOT_ROW["ticker"]
PLOT_CT = PLOT_ROW["ct"]
data = raw_kline.filter((pl.col("ticker") == PLOT_TICKER) & (pl.col("ct") == PLOT_CT)).head(500)
plot_data = (
    col
    .with_cols(
        col("close").mean().rolling(30).alias("ma30"),
        (col("close") / col("close").shift(1) - col.lit(1.0)).alias("ret"),
        (col("close") > col("open")).alias("up"),
    )
    .calc_data(data)
)
print("plot contract:", PLOT_TICKER, PLOT_CT)
plot_data.head(5)


ticker,datetime,open,high,low,close,volume,is_finished,ma30,ret,up
str,datetime[ms],f64,f64,f64,f64,f64,bool,f64,f64,bool
"""au""",2022-07-02 00:01:00,390.160004,390.160004,390.059998,390.119995,81.0,true,null,-0.00128,false
"""au""",2022-07-02 00:02:00.500,390.119995,390.140015,390.079987,390.140015,52.0,true,null,-0.001229,true
"""au""",2022-07-02 00:03:00,390.140015,390.200012,390.119995,390.200012,50.0,true,null,-0.001075,true
"""au""",2022-07-02 00:04:01,390.200012,390.220001,390.140015,390.160004,61.0,true,null,-0.001178,false
"""au""",2022-07-02 00:05:00.500,390.140015,390.140015,390.079987,390.100006,41.0,true,null,-0.001331,false


## 2. line / area / step_line

line 输入至少两列：

```text
x, y1, y2, ...
```

这里 `datetime` 是 x，`close` 和 `ma30` 是两条 y 线。area 同样是 x + y，只是把线下区域填充出来，更适合展示收益、回撤、成交量占比等。


In [3]:
line_rt = col("datetime", "close", "ma30").monitor("line", show_axis_label=True).line().runtime()
line_rt.plot(plot_data, open_in_jupyter=True, auto_open=False, height=420)


In [4]:
area_rt = col("datetime", "ret").monitor("area", show_axis_label=True).area().runtime()
area_rt.plot(plot_data, open_in_jupyter=True, auto_open=False, height=360)


## 3. kline + mark + fill

K 线图最少需要 5 列：

```text
datetime, open, high, low, close
```

mark 图层一般需要：

```text
x, y, flag
```

当 `flag=True` 时，在 `(x, y)` 位置画标记。下面把上涨 bar 画成绿色三角，并把 `ma30` 叠在同一个 `price` slot 里。


In [5]:
kline_expr = col(
    col("datetime", "open", "high", "low", "close").monitor("price").kline(),
    col("datetime", "close", "up").monitor("price").mark(mark_shape.triangle_up, color="lime", width=0.4),
    col("datetime", "ma30").monitor("price").line(),
).monitor.make_monitor("black").monitor.add_grid([["price"]]).runtime()

kline_expr.plot(plot_data, open_in_jupyter=True, auto_open=False, height=520)


## 4. bar 和 table

bar 输入是：

```text
category, value1, value2, ...
```

适合展示不同品种、不同参数、不同策略之间的数值对比。

table 输入可以是任意列，适合展示统计表、参数表、优化结果表。


In [6]:
bar_data = (
    col("close", "volume")
    .mean()
    .group_by("ticker")
    .batch.sort("ticker")
    .calc_data(pl.read_parquet(DATA_KLINE).head(50_000))
)

bar_rt = col("ticker", "close", "volume").monitor("bar", show_axis_label=True).bar().runtime()
bar_rt.plot(bar_data, open_in_jupyter=True, auto_open=False, height=420)


In [7]:
table_rt = col.all.monitor("summary", show_axis_label=True).table().runtime()
table_rt.plot(bar_data, open_in_jupyter=True, auto_open=False, height=360)


## 5. scatter

scatter 输入至少两列：

```text
x, y
```

后续列可以作为 hover 或交互选择时的上下文。散点图适合看两个变量之间的关系，比如成交量和收益、因子值和未来收益。


In [8]:
scatter_rt = col("volume", "ret", "datetime", "close").monitor("scatter", show_axis_label=True).scatter(size=0.04).runtime()
scatter_rt.plot(plot_data.filter(pl.col("ret").is_not_null()), open_in_jupyter=True, auto_open=False, height=460)


## 6. heatmap

heatmap 输入至少三列：

```text
x_label, y_label, value
```

适合展示二维参数平面、相关性矩阵、分组统计矩阵。


In [9]:
heat_data = pl.DataFrame({
    "x": [f"x{i}" for i in range(1, 6) for _ in range(4)],
    "y": [f"y{j}" for _ in range(5) for j in range(1, 5)],
    "value": [((i * 7 + j * 3) % 17) / 16 for i in range(1, 6) for j in range(1, 5)],
})

heat_rt = col("x", "y", "value").monitor("heatmap", show_axis_label=True).heatmap().runtime()
heat_rt.plot(heat_data, open_in_jupyter=True, auto_open=False, height=420)


## 7. violin

violin 不是直接吃原始样本，而是推荐吃：

```text
label + batch.violin_profile() 输出的 profile 列
```

原因：如果图形交互时每次都从原始样本重算分布，拖动/缩放会慢，而且可能因为可见窗口变化导致分布看起来变了。先算 profile 后，monitor 只负责画图，分布是稳定的。


In [10]:
violin_source = pl.DataFrame({
    "elapsed": [i % 6 + 1 for i in range(240)],
    "ret": [((i * 17) % 41 - 20) / 1000 + (i % 6) * 0.002 for i in range(240)],
})

violin_profile = (
    col("ret")
    .batch.violin_profile(lower_bound=0.05, up_bound=0.95)
    .group_by("elapsed")
    .batch.sort("elapsed")
    .calc_data(violin_source)
)

violin_rt = col.all.monitor("violin", show_axis_label=True).violin().runtime()
violin_rt.plot(violin_profile, open_in_jupyter=True, auto_open=False, height=460)


## 8. 多图布局：make_monitor + add_grid

多个 plot layer 可以放进同一个 monitor slot；`add_grid` 控制页面网格。


In [11]:
dashboard = col(
    col("datetime", "open", "high", "low", "close").monitor("price").kline(),
    col("datetime", "ma30").monitor("price").line(),
    col("datetime", "volume").monitor("volume").bar(),
    col("datetime", "ret").monitor("ret").line(),
).monitor.make_monitor("black").monitor.add_grid([
    ["price", "price"],
    ["volume", "ret"],
]).runtime()

dashboard.plot(plot_data, open_in_jupyter=True, auto_open=False, height=720)


## 9. monitor 公共参数

常用公共参数：

- `slot_id`：`monitor("price")` 里的图层槽位；
- `y_axis`：同一 slot 里的第几个 y 轴；
- `y_ratio`：子图高度比例，负数常用于把填充/回撤放到下方；
- `show_axis_label=True`：是否显示轴名称；
- `over_type="stack"/"split"/"select"`：部分图类型支持按 over 分组堆叠、拆分、选择。


In [12]:
pl.DataFrame({
    "parameter": ["slot_id", "y_axis", "y_ratio", "show_axis_label", "over_type"],
    "example": ["monitor('price')", "monitor('price', y_axis=1)", "monitor('dd', y_ratio=-0.3)", "monitor(show_axis_label=False)", "monitor(over_type='stack')"],
    "meaning": ["同一槽位叠加图层", "同一图里使用额外 y 轴", "控制高度或下方子区域", "隐藏/显示轴标题", "按 over 分组交互展示"],
})


parameter,example,meaning
str,str,str
"""slot_id""","""monitor('price')""","""同一槽位叠加图层"""
"""y_axis""","""monitor('price', y_axis=1)""","""同一图里使用额外 y 轴"""
"""y_ratio""","""monitor('dd', y_ratio=-0.3)""","""控制高度或下方子区域"""
"""show_axis_label""","""monitor(show_axis_label=False)""","""隐藏/显示轴标题"""
"""over_type""","""monitor(over_type='stack')""","""按 over 分组交互展示"""
